# Complete Pipeline: PPI Inhibitor Prediction with Graph Neural Networks

This notebook provides a complete end-to-end pipeline for predicting small-molecule inhibitors of protein-protein interactions using Graph Neural Networks.

## Pipeline Overview:
1. **Environment Setup** - Install dependencies and configure GPU
2. **Data Loading** - Load protein structures (PDB), compounds (SMILES), and labels
3. **Feature Extraction** - Process 3D structures and molecular fingerprints
4. **Model Architecture** - Define GNN layers and fusion network
5. **Training** - Cross-validation with Leave-One-Complex-Out (LOCO)
6. **Evaluation** - Calculate metrics and visualize results

**Key Innovation:** First method to predict whether a specific compound inhibits a given protein complex in a targeted manner.

## 1. Environment Setup and Dependencies

In [ ]:
# Install required packages
import sys
import subprocess

def install_packages():
    """Install all required packages"""
    packages = ['biopython', 'rdkit', 'torch', 'scikit-learn', 'numpy', 'pandas', 'matplotlib', 'tqdm']
    for package in packages:
        try:
            __import__(package)
            print(f"✓ {package} already installed")
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_packages()

In [ ]:
# Import all required libraries
import warnings
warnings.filterwarnings('ignore')

# Biology and Chemistry
from Bio.PDB import *
from Bio.PDB.NeighborSearch import NeighborSearch
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.utils.data.sampler import WeightedRandomSampler
from torch.autograd import Variable

# Machine Learning
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, 
    average_precision_score, precision_score, recall_score, auc
)

# Data Processing
import numpy as np
import pandas as pd
import pickle
import glob
from tqdm import tqdm

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Check CUDA availability
USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda:0" if USE_CUDA else "cpu")

if USE_CUDA:
    print(f"✓ CUDA is available. Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("⚠ CUDA is not available. Using CPU (training will be slow)")

print(f"\nUsing device: {device}")

## 2. Utility Functions

In [ ]:
# PyTorch utility functions
def cuda(v):
    """Move tensor to GPU if available"""
    if USE_CUDA:
        return v.cuda()
    return v

def toTensor(v, dtype=torch.float, requires_grad=False):
    """Convert numpy array or list to PyTorch tensor"""
    return cuda(Variable(torch.tensor(v)).type(dtype).requires_grad_(requires_grad))

def toNumpy(v):
    """Convert PyTorch tensor to numpy array"""
    if USE_CUDA:
        return v.detach().cpu().numpy()
    return v.detach().numpy()

print("✓ Utility functions defined")

## 3. Protein Structure Processing

These functions convert PDB files into graph representations suitable for GNN input.

- **atom1()**: One-hot encodes 13 atom types (C, CA, CB, CG, etc.)
- **res1()**: One-hot encodes 21 amino acid residues
- **neigh1()**: Builds adjacency lists for spatial neighbors (within/across residues)

In [ ]:
def atom1(structure):
    """
    One-hot encode atom types from protein structure.
    
    Args:
        structure: BioPython Structure object
    
    Returns:
        numpy array of shape (N_atoms, 13) with one-hot encoded atom types
    """
    atomslist = np.array(sorted(['C', 'CA', 'CB', 'CG', 'CH2', 'N', 'NH2', 
                                  'OG', 'OH', 'O1', 'O2', 'SE', '1'])).reshape(-1, 1)
    enc = OneHotEncoder(handle_unknown='ignore')
    enc.fit(atomslist)
    
    atom_list = []
    for atom in structure.get_atoms():
        if atom.get_name() in atomslist:
            atom_list.append(atom.get_name())
        else:
            atom_list.append("1")  # Unknown atom type
    
    atoms_onehot = enc.transform(np.array(atom_list).reshape(-1, 1)).toarray()
    return atoms_onehot


def res1(structure):
    """
    One-hot encode residue types from protein structure.
    
    Args:
        structure: BioPython Structure object
    
    Returns:
        numpy array of shape (N_atoms, 21) with one-hot encoded residue types
    """
    residuelist = np.array(sorted(['ALA', 'ARG', 'ASN', 'ASP', 'GLN', 'GLU', 'GLY', 
                                    'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 
                                    'THR', 'TRP', 'TYR', 'VAL', 'CYS', 'HIS', '1'])).reshape(-1, 1)
    encr = OneHotEncoder(handle_unknown='ignore')
    encr.fit(residuelist)
    
    residue_list = []
    for atom in structure.get_atoms():
        res_name = atom.get_parent().get_resname()
        if res_name in residuelist:
            residue_list.append(res_name)
        else:
            residue_list.append("1")  # Unknown residue
    
    res_onehot = encr.transform(np.array(residue_list).reshape(-1, 1)).toarray()
    return res_onehot


def neigh1(structure, cutoff_distance=6.0, max_neighbors=10):
    """
    Calculate spatial neighbors for each atom within the structure.
    
    Args:
        structure: BioPython Structure object
        cutoff_distance: Maximum distance (Å) for neighbor search
        max_neighbors: Maximum number of neighbors per residue type
    
    Returns:
        neigh_same_res: Array of shape (N_atoms, 10) with indices of neighbors in same residue
        neigh_diff_res: Array of shape (N_atoms, 10) with indices of neighbors in different residues
    """
    atom_list = np.array([atom for atom in structure.get_atoms()])
    
    # Find all neighbors within cutoff distance
    ns = NeighborSearch(atom_list)
    neighbour_list = ns.search_all(cutoff_distance, level="A")
    neighbour_list = np.array(neighbour_list)
    
    # Calculate distances and sort
    dist = np.array([nl[0] - nl[1] for nl in neighbour_list])
    place = np.argsort(dist)
    sorted_neighbour_list = neighbour_list[place]
    
    # Map atom serial numbers to indices
    old_atom_number = np.array([atom.get_serial_number() for atom in atom_list])
    old_residue_number = np.array([atom.get_parent().get_id()[1] for atom in atom_list])
    
    total_atoms = len(atom_list)
    
    # Initialize neighbor arrays with -1 (no neighbor)
    neigh_same_res = np.full((total_atoms, max_neighbors), -1, dtype=np.int32)
    neigh_diff_res = np.full((total_atoms, max_neighbors), -1, dtype=np.int32)
    same_flag = [0] * total_atoms
    diff_flag = [0] * total_atoms
    
    # Populate neighbor lists
    for source_atom, neigh_atom in sorted_neighbour_list:
        source_atom_id = source_atom.get_serial_number()
        neigh_atom_id = neigh_atom.get_serial_number()
        source_atom_res = source_atom.get_parent().get_id()[1]
        neigh_atom_res = neigh_atom.get_parent().get_id()[1]
        
        # Find indices in original atom array
        temp_index1 = np.where(source_atom_id == old_atom_number)[0]
        temp_index2 = np.where(neigh_atom_id == old_atom_number)[0]
        
        source_index = None
        neigh_index = None
        
        for i1 in temp_index1:
            if old_residue_number[i1] == source_atom_res:
                source_index = i1
                break
        
        for i1 in temp_index2:
            if old_residue_number[i1] == neigh_atom_res:
                neigh_index = i1
                break
        
        if source_index is None or neigh_index is None:
            continue
        
        # Same residue neighbors
        if source_atom_res == neigh_atom_res:
            if same_flag[source_index] < max_neighbors:
                neigh_same_res[source_index][same_flag[source_index]] = neigh_index
                same_flag[source_index] += 1
            
            if same_flag[neigh_index] < max_neighbors:
                neigh_same_res[neigh_index][same_flag[neigh_index]] = source_index
                same_flag[neigh_index] += 1
        
        # Different residue neighbors
        else:
            if diff_flag[source_index] < max_neighbors:
                neigh_diff_res[source_index][diff_flag[source_index]] = neigh_index
                diff_flag[source_index] += 1
            
            if diff_flag[neigh_index] < max_neighbors:
                neigh_diff_res[neigh_index][diff_flag[neigh_index]] = source_index
                diff_flag[neigh_index] += 1
    
    return neigh_same_res, neigh_diff_res


def process_proteins(protein_ids, pdb_location):
    """
    Process multiple PDB files and create graph representations.
    
    Args:
        protein_ids: List of protein IDs (without .pdb extension)
        pdb_location: Directory containing PDB files
    
    Returns:
        Dictionary mapping protein IDs to graph data [atoms, residues, same_neighbors, diff_neighbors]
    """
    protein_data_dict = {}
    parser = PDBParser(QUIET=True)
    
    for i, pid in enumerate(tqdm(protein_ids, desc="Processing PDB files")):
        pid_clean = pid.split('.pdb')[0]
        pdb_path = f"{pdb_location}/{pid_clean}.pdb"
        
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                structure = parser.get_structure("", pdb_path)
            
            # Extract features
            one_hot_atom = atom1(structure)
            one_hot_res = res1(structure)
            neigh_same_res, neigh_diff_res = neigh1(structure)
            
            # Convert to PyTorch tensors
            one_hot_atom = torch.tensor(one_hot_atom, dtype=torch.float32).to(device)
            one_hot_res = torch.tensor(one_hot_res, dtype=torch.float32).to(device)
            neigh_same_res = torch.tensor(neigh_same_res).to(device).long()
            neigh_diff_res = torch.tensor(neigh_diff_res).to(device).long()
            
            protein_data_dict[pid_clean] = [one_hot_atom, one_hot_res, neigh_same_res, neigh_diff_res]
        
        except Exception as e:
            print(f"Error processing {pid_clean}: {e}")
    
    return protein_data_dict

print("✓ Protein processing functions defined")

## 4. Compound Processing

Convert SMILES strings to molecular fingerprints using RDKit.

In [ ]:
def smiles_to_fingerprint(smiles, radius=2, n_bits=2048):
    """
    Convert SMILES string to Morgan fingerprint.
    
    Args:
        smiles: SMILES string representation of molecule
        radius: Radius for Morgan fingerprint (default: 2)
        n_bits: Number of bits in fingerprint (default: 2048)
    
    Returns:
        Numpy array of molecular fingerprint
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(n_bits)
        
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        arr = np.zeros((1,))
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr
    except:
        return np.zeros(n_bits)


def process_compounds(compound_smiles_dict):
    """
    Process multiple compounds and generate fingerprints.
    
    Args:
        compound_smiles_dict: Dictionary mapping compound IDs to SMILES strings
    
    Returns:
        Dictionary mapping compound IDs to fingerprint arrays
    """
    compound_fp_dict = {}
    
    for cid, smiles in tqdm(compound_smiles_dict.items(), desc="Processing compounds"):
        fp = smiles_to_fingerprint(smiles)
        compound_fp_dict[cid] = fp
    
    return compound_fp_dict

print("✓ Compound processing functions defined")

## 5. Custom Datasets and Samplers

Implement balanced sampling to handle class imbalance (positive vs negative examples).

In [ ]:
class CustomDataset(Dataset):
    """Simple dataset wrapper for protein-compound pairs."""
    
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


class BinaryBalancedSampler(Sampler):
    """
    A PyTorch Sampler that returns batches with equal positive and negative examples.
    Oversamples the minority class to balance with the majority class.
    """
    
    def __init__(self, class_vector, batch_size=10):
        self.batch_size = batch_size
        self.class_vector = np.array(class_vector)
        
        # Find majority and minority classes
        U, C = np.unique(self.class_vector, return_counts=True)
        M = U[np.argmax(C)]  # Majority class
        
        Midx = np.nonzero(self.class_vector == M)[0]  # Majority indices
        midx = np.nonzero(self.class_vector != M)[0]  # Minority indices
        
        # Oversample minority to match majority
        midx_ = np.random.choice(midx, size=len(Midx), replace=True)
        
        self.YY = np.array(list(self.class_vector[Midx]) + list(self.class_vector[midx_]))
        self.idx = np.array(list(Midx) + list(midx_))
        
        self.n_splits = int(np.ceil(len(self.idx) / self.batch_size))
        self.equivalent_epochs = len(self.idx) / len(self.class_vector)
        
        print(f"Balanced sampler: {self.equivalent_epochs:.2f} equivalent epochs per iteration")
    
    def gen_sample_array(self):
        """Generate balanced batches using stratified sampling."""
        from sklearn.model_selection import StratifiedKFold
        
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True)
        for _, ttidx in skf.split(self.idx, self.YY):
            yield self.idx[ttidx]
    
    def __iter__(self):
        return iter(self.gen_sample_array())
    
    def __len__(self):
        return self.n_splits

print("✓ Dataset and sampler classes defined")

## 6. Graph Neural Network Architecture

Define the GNN model architecture:
- **GNN_First_Layer**: Initial layer processing atomic and residue features
- **GNN_Layer**: Convolutional layers aggregating neighbor information
- **GNN**: Complete GNN model with 3 layers + pooling
- **IPPI_MLP_Net**: MLP for fusing GNN protein features + compound fingerprints

In [ ]:
class GNN_First_Layer(nn.Module):
    """
    First GNN layer that processes atomic and residue features.
    Combines atom type, residue type, and spatial neighbor information.
    """
    
    def __init__(self, filters=512, n_atom_types=13, n_residue_types=21):
        super(GNN_First_Layer, self).__init__()
        self.filters = filters
        
        # Learnable weight matrices
        self.Wv = nn.Parameter(torch.randn(n_atom_types, filters, device=device, requires_grad=True))
        self.Wr = nn.Parameter(torch.randn(n_residue_types, filters, device=device, requires_grad=True))
        self.Wsr = nn.Parameter(torch.randn(n_atom_types, filters, device=device, requires_grad=True))
        self.Wdr = nn.Parameter(torch.randn(n_atom_types, filters, device=device, requires_grad=True))
    
    def forward(self, x):
        atoms, residues, same_neigh, diff_neigh = x
        
        # Node signals
        node_signals = atoms @ self.Wv
        residue_signals = residues @ self.Wr
        
        # Neighbor aggregation
        neigh_signals_same = atoms @ self.Wsr
        neigh_signals_diff = atoms @ self.Wdr
        
        # Mask for valid neighbors (>-1)
        unsqueezed_same_neigh_indicator = (same_neigh > -1).unsqueeze(2)
        unsqueezed_diff_neigh_indicator = (diff_neigh > -1).unsqueeze(2)
        
        # Gather neighbor features
        same_neigh_features = neigh_signals_same[same_neigh] * unsqueezed_same_neigh_indicator
        diff_neigh_features = neigh_signals_diff[diff_neigh] * unsqueezed_diff_neigh_indicator
        
        # Normalize by number of neighbors
        same_norm = torch.sum(same_neigh > -1, 1).unsqueeze(1).type(torch.float)
        diff_norm = torch.sum(diff_neigh > -1, 1).unsqueeze(1).type(torch.float)
        
        # Prevent division by zero
        same_norm[same_norm == 0] = 1
        diff_norm[diff_norm == 0] = 1
        
        neigh_same_atoms_signal = torch.sum(same_neigh_features, axis=1) / same_norm
        neigh_diff_atoms_signal = torch.sum(diff_neigh_features, axis=1) / diff_norm
        
        # Combine all signals with ReLU activation
        final_res = torch.relu(node_signals + residue_signals + 
                               neigh_same_atoms_signal + neigh_diff_atoms_signal)
        
        return final_res, same_neigh, diff_neigh


class GNN_Layer(nn.Module):
    """
    Subsequent GNN layers for deeper feature extraction.
    Aggregates information from same-residue and different-residue neighbors.
    """
    
    def __init__(self, filters, v_feats):
        super(GNN_Layer, self).__init__()
        self.v_feats = v_feats
        self.filters = filters
        
        # Learnable weight matrices
        self.Wsv = nn.Parameter(torch.randn(v_feats, filters, device=device, requires_grad=True))
        self.Wdr = nn.Parameter(torch.randn(v_feats, filters, device=device, requires_grad=True))
        self.Wsr = nn.Parameter(torch.randn(v_feats, filters, device=device, requires_grad=True))
    
    def forward(self, x):
        Z, same_neigh, diff_neigh = x
        
        # Transform node features
        node_signals = Z @ self.Wsv
        neigh_signals_same = Z @ self.Wsr
        neigh_signals_diff = Z @ self.Wdr
        
        # Mask for valid neighbors
        unsqueezed_same_neigh_indicator = (same_neigh > -1).unsqueeze(2)
        unsqueezed_diff_neigh_indicator = (diff_neigh > -1).unsqueeze(2)
        
        # Gather and aggregate neighbor features
        same_neigh_features = neigh_signals_same[same_neigh] * unsqueezed_same_neigh_indicator
        diff_neigh_features = neigh_signals_diff[diff_neigh] * unsqueezed_diff_neigh_indicator
        
        # Normalize
        same_norm = torch.sum(same_neigh > -1, 1).unsqueeze(1).type(torch.float)
        diff_norm = torch.sum(diff_neigh > -1, 1).unsqueeze(1).type(torch.float)
        same_norm[same_norm == 0] = 1
        diff_norm[diff_norm == 0] = 1
        
        neigh_same_atoms_signal = torch.sum(same_neigh_features, axis=1) / same_norm
        neigh_diff_atoms_signal = torch.sum(diff_neigh_features, axis=1) / diff_norm
        
        # Combine with ReLU
        final_res = torch.relu(node_signals + neigh_same_atoms_signal + neigh_diff_atoms_signal)
        
        return final_res, same_neigh, diff_neigh


class GNN(nn.Module):
    """
    Complete GNN model with 3 convolutional layers.
    Output: Fixed-size protein representation via global pooling.
    """
    
    def __init__(self):
        super(GNN, self).__init__()
        self.conv1 = GNN_First_Layer(filters=512)
        self.conv2 = GNN_Layer(v_feats=512, filters=1024)
        self.conv3 = GNN_Layer(v_feats=1024, filters=512)
    
    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)
        
        # Global sum pooling
        x = x3[0]
        x = torch.sum(x, axis=0).view(1, -1)
        
        # L2 normalization
        x = F.normalize(x)
        
        return x


class IPPI_MLP_Net(nn.Module):
    """
    Multi-Layer Perceptron for fusion of:
    - GNN protein features (512-dim)
    - Interface features (280-dim)
    - Compound fingerprints (2048-dim)
    
    Total input: 2840 dimensions
    Output: Binary classification (inhibitor vs non-inhibitor)
    """
    
    def __init__(self, input_dim=2840):
        super(IPPI_MLP_Net, self).__init__()
        self.fc1 = nn.Linear(input_dim, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 100)
        self.fc6 = nn.Linear(100, 1)
    
    def forward(self, protein_features, compound_features, interface_features):
        # Concatenate all features
        protein_all_features = torch.hstack((protein_features, interface_features))
        pc_features = torch.hstack((protein_all_features, compound_features))
        
        # Forward pass through MLP
        x = torch.tanh(self.fc1(pc_features))
        x = torch.tanh(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc6(x)
        
        return x

print("✓ GNN architecture defined")

## 7. Data Loading

Load all required data:
- Training examples (protein-compound pairs with labels)
- Pre-computed features (interface features, compound fingerprints)
- Protein structures (from PDB files)

In [ ]:
def load_training_data(data_file):
    """
    Load training examples from file.
    
    File format:
    RootComplex TargetComplex CompoundID Label
    
    Returns:
        complexes: List of complex names
        compounds: List of compound IDs
        labels: List of labels (1.0 for inhibitor, 0.0 for non-inhibitor)
        test_pos_complexes: Root complex for grouping (for cross-validation)
    """
    with open(data_file) as f:
        lines = f.readlines()
    
    test_pos_complexes = []
    complexes = []
    compounds = []
    labels = []
    
    for line in tqdm(lines, desc="Loading training data"):
        parts = line.strip().split()
        
        if len(parts) == 4:
            test_pos_comp, complex_name, compound_name, label = parts
        else:
            # Handle compound names with spaces
            test_pos_comp = parts[0]
            complex_name = parts[1]
            compound_name = ' '.join(parts[2:-1])
            label = parts[-1]
        
        test_pos_complexes.append(test_pos_comp)
        complexes.append(complex_name)
        compounds.append(compound_name)
        labels.append(float(label))
    
    return test_pos_complexes, complexes, compounds, labels


# Configuration
DATA_DIR = './Data/'
FEATURES_DIR = './Features/'
PDB_DIR = './Data/DBD5/'
MODELS_DIR = './trained_models/'

# Create output directory if needed
import os
os.makedirs(MODELS_DIR, exist_ok=True)

print("\n=== Loading Data ===")

# Load training examples
test_pos_complexes, complexes, compounds, labels = load_training_data(
    DATA_DIR + 'WriteAllexamplesRandomBindersIdsAll_24JAN_Binary.txt'
)

print(f"✓ Loaded {len(labels)} training examples")
print(f"  - Positive examples: {sum(labels)} ({sum(labels)/len(labels)*100:.1f}%)")
print(f"  - Negative examples: {len(labels)-sum(labels)} ({(1-sum(labels)/len(labels))*100:.1f}%)")

# Load pre-computed features if available
print("\nLoading pre-computed features...")
try:
    # Interface features
    pos_interface_dict = pickle.load(open(FEATURES_DIR + 'Pos_seqandInterfaceF_dict.npy', 'rb'))
    dbds_interface_dict = pickle.load(open(FEATURES_DIR + 'NewUbench5InterfaceandSeq_dict.npy', 'rb'))
    interface_features_dict = {**pos_interface_dict, **dbds_interface_dict}
    
    # Simplify keys (remove chain information)
    simplified_interface_dict = {}
    for key in interface_features_dict:
        if len(key.split('_')) > 1:
            simple_key = key.split('_')[0]
            simplified_interface_dict[simple_key] = interface_features_dict[key]
        else:
            simplified_interface_dict[key] = interface_features_dict[key]
    
    print(f"✓ Loaded interface features for {len(simplified_interface_dict)} complexes")
    
    # Compound fingerprints
    compound_fp_dict = pickle.load(open(FEATURES_DIR + 'Compound_Fingerprint_Features_Dict.npy', 'rb'))
    print(f"✓ Loaded fingerprints for {len(compound_fp_dict)} compounds")
    
except FileNotFoundError as e:
    print(f"⚠ Warning: Could not load pre-computed features: {e}")
    print("  You will need to generate features from raw data.")
    simplified_interface_dict = {}
    compound_fp_dict = {}

print("\n✓ Data loading complete")

## 8. Cross-Validation Setup

Use Leave-One-Complex-Out (LOCO) cross-validation:
- Train on 22 complexes
- Test on 1 held-out complex
- Repeat for all 23 complexes

In [ ]:
# Prepare data for cross-validation
complexes = np.array(complexes)
compounds = np.array(compounds)
labels = np.array(labels)
test_pos_complexes = np.array(test_pos_complexes)

# Create dictionary of all examples
all_data = list(zip(test_pos_complexes, zip(complexes, compounds)))
all_examples = dict(zip(all_data, labels))

# Group by root complex for LOCO cross-validation
groups = [k.split('_')[0] for k in test_pos_complexes]
unique_complexes = sorted(set(groups))

print(f"\n=== Cross-Validation Setup ===")
print(f"Number of unique complexes: {len(unique_complexes)}")
print(f"Complexes: {', '.join(unique_complexes)}")

# Setup GroupKFold
groups_df = pd.DataFrame(groups)
gkf = GroupKFold(n_splits=len(unique_complexes))

print(f"\n✓ Using Leave-One-Complex-Out cross-validation ({len(unique_complexes)} folds)")

## 9. Training Loop

Train the GNN model using LOCO cross-validation.
For each fold:
1. Split data into train/test
2. Standardize features
3. Train GNN + MLP for multiple epochs
4. Evaluate on held-out complex
5. Save best model based on validation AUC-ROC

In [ ]:
def train_model(train_data, test_data, protein_data_dict, interface_dict, compound_dict, 
                complex_name, n_epochs=5, batch_size=1024, learning_rate=0.001):
    """
    Train GNN model for one fold of cross-validation.
    
    Args:
        train_data: List of training examples
        test_data: List of test examples
        protein_data_dict: Dictionary of protein graph data
        interface_dict: Dictionary of interface features
        compound_dict: Dictionary of compound fingerprints
        complex_name: Name of test complex
        n_epochs: Number of training epochs
        batch_size: Batch size
        learning_rate: Learning rate
    
    Returns:
        best_models: Tuple of (GNN state_dict, MLP state_dict)
        test_scores: Predictions on test set
        test_targets: True labels for test set
    """
    
    print(f"\n{'='*60}")
    print(f"Training for test complex: {complex_name}")
    print(f"{'='*60}")
    
    # Prepare training data
    compound_train = []
    protein_train = []
    y_train = []
    compound_train_names = []
    protein_train_names = []
    
    for t in train_data:
        complex_id = t[1][0].split('_')[0]
        compound_id = t[1][1]
        
        if complex_id in interface_dict and compound_id in compound_dict:
            protein_train.append(interface_dict[complex_id])
            protein_train_names.append(complex_id)
            compound_train.append(compound_dict[compound_id])
            compound_train_names.append(compound_id)
            y_train.append(all_examples[(t[0], t[1])])
    
    # Prepare test data
    compound_test = []
    protein_test = []
    y_test = []
    compound_test_names = []
    protein_test_names = []
    
    for t in test_data:
        complex_id = t[1][0].split('_')[0]
        compound_id = t[1][1]
        
        if complex_id in interface_dict and compound_id in compound_dict:
            protein_test.append(interface_dict[complex_id])
            protein_test_names.append(complex_id)
            compound_test.append(compound_dict[compound_id])
            compound_test_names.append(compound_id)
            y_test.append(all_examples[(t[0], t[1])])
    
    print(f"Training examples: {len(y_train)}")
    print(f"  Positive: {sum(y_train)} ({sum(y_train)/len(y_train)*100:.1f}%)")
    print(f"Test examples: {len(y_test)}")
    print(f"  Positive: {sum(y_test)} ({sum(y_test)/len(y_test)*100:.1f}%)")
    
    # Standardize features
    protein_scaler = StandardScaler().fit(protein_train)
    compound_scaler = StandardScaler().fit(compound_train)
    
    compound_train = compound_scaler.transform(compound_train)
    protein_train = protein_scaler.transform(protein_train)
    protein_test = protein_scaler.transform(protein_test)
    compound_test = compound_scaler.transform(compound_test)
    
    # Create dictionaries for easy lookup
    protein_train_dict = dict(zip(protein_train_names, torch.FloatTensor(protein_train).to(device)))
    compound_train_dict = dict(zip(compound_train_names, torch.FloatTensor(compound_train).to(device)))
    protein_test_dict = dict(zip(protein_test_names, torch.FloatTensor(protein_test).to(device)))
    compound_test_dict = dict(zip(compound_test_names, torch.FloatTensor(compound_test).to(device)))
    
    # Initialize models
    gnn_model = GNN().to(device)
    mlp_model = IPPI_MLP_Net().to(device)
    
    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(
        list(gnn_model.parameters()) + list(mlp_model.parameters()),
        lr=learning_rate
    )
    
    # Create data loaders
    y_train = np.array(y_train)
    train_dataset = CustomDataset(train_data[:len(y_train), 1], y_train.astype('int'))
    train_sampler = BinaryBalancedSampler(y_train.astype('int'), batch_size)
    train_loader = DataLoader(train_dataset, batch_sampler=train_sampler)
    
    test_dataset = CustomDataset(test_data[:len(y_test), 1], np.array(y_test).astype('int'))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Training loop
    best_auc = 0.0
    best_models = None
    loss_history = []
    
    for epoch in range(n_epochs):
        print(f"\nEpoch {epoch+1}/{n_epochs}")
        
        # Training phase
        gnn_model.train()
        mlp_model.train()
        epoch_losses = []
        
        for (batch_proteins, batch_compounds), batch_labels in tqdm(train_loader, desc="Training"):
            # Get unique proteins in batch
            protein_ids = [p.split('_')[0] for p in batch_proteins]
            unique_proteins = list(set(protein_ids))
            
            # Process each unique protein through GNN once
            gnn_features_dict = {}
            for pid in unique_proteins:
                if pid in protein_data_dict:
                    gnn_features_dict[pid] = gnn_model(protein_data_dict[pid])
            
            # Gather features for batch
            gnn_features = torch.vstack([gnn_features_dict[p] for p in protein_ids if p in gnn_features_dict])
            interface_features = torch.vstack([protein_train_dict[p] for p in protein_ids if p in protein_train_dict])
            compound_features = torch.vstack([compound_train_dict[c] for c in batch_compounds if c in compound_train_dict])
            
            # Forward pass
            output = mlp_model(gnn_features, compound_features, interface_features)
            loss = criterion(output.flatten(), batch_labels.float().to(device))
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_losses.append(loss.item())
        
        avg_loss = np.mean(epoch_losses)
        loss_history.append(avg_loss)
        print(f"Average training loss: {avg_loss:.4f}")
        
        # Validation phase
        gnn_model.eval()
        mlp_model.eval()
        
        test_scores = []
        test_targets = []
        
        with torch.no_grad():
            for (batch_proteins, batch_compounds), batch_labels in test_loader:
                protein_ids = [p.split('_')[0] for p in batch_proteins]
                unique_proteins = list(set(protein_ids))
                
                gnn_features_dict = {}
                for pid in unique_proteins:
                    if pid in protein_data_dict:
                        gnn_features_dict[pid] = gnn_model(protein_data_dict[pid])
                
                gnn_features = torch.vstack([gnn_features_dict[p] for p in protein_ids if p in gnn_features_dict])
                interface_features = torch.vstack([protein_test_dict[p] for p in protein_ids if p in protein_test_dict])
                compound_features = torch.vstack([compound_test_dict[c] for c in batch_compounds if c in compound_test_dict])
                
                output = mlp_model(gnn_features, compound_features, interface_features)
                test_scores.extend(output.cpu().flatten().numpy())
                test_targets.extend(batch_labels.cpu().flatten().numpy())
        
        # Calculate metrics
        auc_roc = roc_auc_score(test_targets, test_scores)
        auc_pr = average_precision_score(test_targets, test_scores)
        
        print(f"Validation AUC-ROC: {auc_roc:.4f}")
        print(f"Validation AUC-PR: {auc_pr:.4f}")
        
        # Save best model
        if auc_roc > best_auc:
            best_auc = auc_roc
            best_models = (gnn_model.state_dict().copy(), mlp_model.state_dict().copy())
            print(f"✓ New best model (AUC-ROC: {best_auc:.4f})")
    
    print(f"\nBest validation AUC-ROC: {best_auc:.4f}")
    
    # Load best model and get final predictions
    gnn_model.load_state_dict(best_models[0])
    mlp_model.load_state_dict(best_models[1])
    
    gnn_model.eval()
    mlp_model.eval()
    
    final_scores = []
    final_targets = []
    
    with torch.no_grad():
        for (batch_proteins, batch_compounds), batch_labels in test_loader:
            protein_ids = [p.split('_')[0] for p in batch_proteins]
            unique_proteins = list(set(protein_ids))
            
            gnn_features_dict = {}
            for pid in unique_proteins:
                if pid in protein_data_dict:
                    gnn_features_dict[pid] = gnn_model(protein_data_dict[pid])
            
            gnn_features = torch.vstack([gnn_features_dict[p] for p in protein_ids if p in gnn_features_dict])
            interface_features = torch.vstack([protein_test_dict[p] for p in protein_ids if p in protein_test_dict])
            compound_features = torch.vstack([compound_test_dict[c] for c in batch_compounds if c in compound_test_dict])
            
            output = mlp_model(gnn_features, compound_features, interface_features)
            final_scores.extend(output.cpu().flatten().numpy())
            final_targets.extend(batch_labels.cpu().flatten().numpy())
    
    return best_models, np.array(final_scores), np.array(final_targets)

print("✓ Training function defined")

## 10. Run Cross-Validation

Execute the complete LOCO cross-validation.

In [ ]:
# Note: This cell requires pre-computed protein features
# If you don't have them, you need to process PDB files first

print("\n" + "="*60)
print("Starting Leave-One-Complex-Out Cross-Validation")
print("="*60)

# Check if we have protein graph data
# In practice, you would load this from pickle files or generate it
print("\n⚠ Note: This example requires pre-computed protein graph data.")
print("  To run the full pipeline, you need to:")
print("  1. Process all PDB files using process_proteins()")
print("  2. Or load pre-computed features from pickle files")
print("\n  Example:")
print("  ```python")
print("  # Option 1: Load pre-computed")
print("  protein_data_dict = pickle.load(open('ProteinData_dict.pickle', 'rb'))")
print("  ")
print("  # Option 2: Process PDB files")
print("  protein_ids = [f.split('.pdb')[0] for f in os.listdir(PDB_DIR) if f.endswith('.pdb')]")
print("  protein_data_dict = process_proteins(protein_ids, PDB_DIR)")
print("  ```")

# Placeholder for demonstration
# Uncomment and modify for actual execution
"""
all_scores = []
all_targets = []
fold_results = []

all_data_array = np.array(all_data)

for fold, (train_idx, test_idx) in enumerate(gkf.split(all_data_array, groups, groups=groups_df)):
    train_data = all_data_array[train_idx]
    test_data = all_data_array[test_idx]
    
    test_complex = test_data[0][0].split('_')[0]
    
    # Train model
    best_models, test_scores, test_targets = train_model(
        train_data, test_data, 
        protein_data_dict,  # Load this!
        simplified_interface_dict,
        compound_fp_dict,
        test_complex,
        n_epochs=5,
        batch_size=1024
    )
    
    # Calculate metrics
    auc_roc = roc_auc_score(test_targets, test_scores)
    auc_pr = average_precision_score(test_targets, test_scores)
    
    fold_results.append({
        'complex': test_complex,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr
    })
    
    all_scores.extend(test_scores)
    all_targets.extend(test_targets)
    
    # Save models
    torch.save(best_models[0], f"{MODELS_DIR}/GNN_model_{test_complex}.pt")
    torch.save(best_models[1], f"{MODELS_DIR}/MLP_model_{test_complex}.pt")
    
    print(f"\nFold {fold+1} complete: {test_complex}")
    print(f"  AUC-ROC: {auc_roc:.4f}")
    print(f"  AUC-PR: {auc_pr:.4f}")

# Save results
np.save(f"{MODELS_DIR}/all_scores.npy", all_scores)
np.save(f"{MODELS_DIR}/all_targets.npy", all_targets)
pd.DataFrame(fold_results).to_csv(f"{MODELS_DIR}/fold_results.csv", index=False)
"""

print("\n✓ Cross-validation code ready (see comments for execution)")

## 11. Evaluation and Visualization

Calculate overall metrics and create visualizations.

In [ ]:
def plot_roc_curve(targets, scores, title="ROC Curve", save_path=None):
    """
    Plot ROC curve.
    
    Args:
        targets: True labels
        scores: Predicted scores
        title: Plot title
        save_path: Path to save figure (optional)
    """
    fpr, tpr, _ = roc_curve(targets, scores)
    auc_score = roc_auc_score(targets, scores)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkblue', lw=2, label=f'AUC-ROC = {auc_score:.3f}')
    plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(alpha=0.3)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()


def plot_pr_curve(targets, scores, title="Precision-Recall Curve", save_path=None):
    """
    Plot Precision-Recall curve.
    
    Args:
        targets: True labels
        scores: Predicted scores
        title: Plot title
        save_path: Path to save figure (optional)
    """
    precision, recall, _ = precision_recall_curve(targets, scores)
    auc_pr = average_precision_score(targets, scores)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='darkgreen', lw=2, label=f'AUC-PR = {auc_pr:.3f}')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(loc="lower left", fontsize=11)
    plt.grid(alpha=0.3)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()


def plot_per_complex_results(results_df, save_path=None):
    """
    Plot per-complex AUC-ROC and AUC-PR scores.
    
    Args:
        results_df: DataFrame with columns ['complex', 'auc_roc', 'auc_pr']
        save_path: Path to save figure (optional)
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Sort by AUC-ROC
    results_sorted = results_df.sort_values('auc_roc', ascending=True)
    
    # AUC-ROC
    ax1.barh(results_sorted['complex'], results_sorted['auc_roc'], color='steelblue')
    ax1.axvline(x=results_sorted['auc_roc'].mean(), color='red', linestyle='--', 
                label=f'Mean: {results_sorted["auc_roc"].mean():.3f}')
    ax1.set_xlabel('AUC-ROC', fontsize=12)
    ax1.set_ylabel('Complex', fontsize=12)
    ax1.set_title('AUC-ROC per Complex', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3, axis='x')
    
    # AUC-PR
    ax2.barh(results_sorted['complex'], results_sorted['auc_pr'], color='seagreen')
    ax2.axvline(x=results_sorted['auc_pr'].mean(), color='red', linestyle='--',
                label=f'Mean: {results_sorted["auc_pr"].mean():.3f}')
    ax2.set_xlabel('AUC-PR', fontsize=12)
    ax2.set_ylabel('Complex', fontsize=12)
    ax2.set_title('AUC-PR per Complex', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3, axis='x')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    plt.show()


# Example: Load and visualize results
# Uncomment when you have results to visualize
"""
# Load results
all_scores = np.load(f"{MODELS_DIR}/all_scores.npy")
all_targets = np.load(f"{MODELS_DIR}/all_targets.npy")
fold_results = pd.read_csv(f"{MODELS_DIR}/fold_results.csv")

# Calculate overall metrics
overall_auc_roc = roc_auc_score(all_targets, all_scores)
overall_auc_pr = average_precision_score(all_targets, all_scores)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Overall AUC-ROC: {overall_auc_roc:.4f}")
print(f"Overall AUC-PR: {overall_auc_pr:.4f}")
print(f"\nPer-fold AUC-ROC: {fold_results['auc_roc'].mean():.4f} ± {fold_results['auc_roc'].std():.4f}")
print(f"Per-fold AUC-PR: {fold_results['auc_pr'].mean():.4f} ± {fold_results['auc_pr'].std():.4f}")

# Plot results
plot_roc_curve(all_targets, all_scores, 
               title="GNN Pipeline - ROC Curve (All Folds)",
               save_path=f"{MODELS_DIR}/roc_curve.png")

plot_pr_curve(all_targets, all_scores,
              title="GNN Pipeline - Precision-Recall Curve (All Folds)",
              save_path=f"{MODELS_DIR}/pr_curve.png")

plot_per_complex_results(fold_results,
                        save_path=f"{MODELS_DIR}/per_complex_results.png")
"""

print("\n✓ Visualization functions defined")

## 12. Summary and Next Steps

This notebook provides a complete pipeline for PPI inhibitor prediction.

### Key Components:
1. **Data Processing**: PDB → Graph representation
2. **Feature Engineering**: Atomic, residue, and spatial features
3. **GNN Architecture**: 3-layer graph convolutional network
4. **Training**: LOCO cross-validation with balanced sampling
5. **Evaluation**: AUC-ROC, AUC-PR metrics

### Expected Performance:
- **AUC-ROC**: ~0.85-0.86
- **AUC-PR**: ~0.43-0.44
- Superior to baseline SVM (0.74) and GearNet (0.83)

### To Run the Full Pipeline:
1. Ensure all data files are in place:
   - `Data/WriteAllexamplesRandomBindersIdsAll_24JAN_Binary.txt`
   - PDB files in `Data/DBD5/` and `Data/Pdb/`
   - Pre-computed features in `Features/`

2. Process protein structures (if not using pre-computed features):
   ```python
   protein_ids = os.listdir(PDB_DIR)
   protein_data_dict = process_proteins(protein_ids, PDB_DIR)
   ```

3. Run cross-validation (Section 10)

4. Visualize results (Section 11)

### Citation:
If you use this code, please cite:
- Original repository: https://github.com/adibayaseen/PPI-Inhibitors
- Paper: [Add publication details when available]

### Contact:
For questions or issues, please open an issue on GitHub.